# Data Lab 8 — Quality & SPC — Control Charts & Cpk
**Certificate 04 · Data Analytics**  |  [← Back to the Lab Hub](../../index.ipynb)

Uses a shared synthetic **factory production log** — one row per part made across 3 presses, 2 shifts, 5 days. pandas, NumPy and matplotlib are pre-installed in Colab, so there is nothing to install.

## Objectives
By the end you will be able to:
- Build an individuals control chart with mean +/- 3 sigma limits.
- Build an X-bar chart from subgroups and flag out-of-control points.
- Compute Cp / Cpk process capability against spec limits — and see why 'in control' is not the same as 'capable'.

## Load & set the spec
Every quality characteristic has a **specification**: a Lower and Upper Spec Limit (LSL/USL) and a target. For oven temperature we use LSL 185, target 200, USL 215 degrees C.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/IF-JL/COEFAM-Labs/main/labs/cert04_data_analytics/data/"
df = pd.read_csv(DATA + "production_log.csv", parse_dates=["timestamp"])
LSL, TARGET, USL = 185, 200, 215
df.head()

## 1 · Individuals control chart (I-chart)
Plot each reading in time order with a centre line (mean) and **control limits** at mean +/- 3 sigma. Control limits come from the *process itself* — they are not the spec limits.

In [ ]:
seq = df[df.machine=="Press_A"].sort_values("timestamp")["oven_temp_c"].reset_index(drop=True).head(150)
cl, sd = seq.mean(), seq.std()
ucl, lcl = cl + 3*sd, cl - 3*sd

plt.figure(figsize=(10,3))
plt.plot(seq.index, seq.values, marker=".", linewidth=.8)
plt.axhline(cl, color="k"); plt.axhline(ucl, color="r", ls="--"); plt.axhline(lcl, color="r", ls="--")
oos = seq[(seq > ucl) | (seq < lcl)]
plt.scatter(oos.index, oos.values, color="red", zorder=5, label="out of control")
plt.title(f"Press_A I-chart  (CL {cl:.1f}, UCL {ucl:.1f}, LCL {lcl:.1f})")
plt.ylabel("deg C"); plt.legend(); plt.show()
print("out-of-control points in this window:", len(oos))

## 2 · X-bar chart (subgroups)
Averaging small **subgroups** (here 5 consecutive parts) smooths noise and tightens the limits to sigma / sqrt(n) — the classic SPC chart for spotting process shifts.

In [ ]:
s = df[df.machine=="Press_C"].sort_values("timestamp")["oven_temp_c"].reset_index(drop=True)
n = 5
xbar = s.groupby(s.index // n).mean()
grand = xbar.mean()
sigma_xbar = s.std() / (n ** 0.5)
ucl, lcl = grand + 3*sigma_xbar, grand - 3*sigma_xbar

plt.figure(figsize=(10,3))
plt.plot(xbar.index, xbar.values, marker="o", linewidth=.7)
plt.axhline(grand, color="k"); plt.axhline(ucl, color="r", ls="--"); plt.axhline(lcl, color="r", ls="--")
bad = xbar[(xbar > ucl) | (xbar < lcl)]
plt.scatter(bad.index, bad.values, color="red", zorder=5, label="out of control")
plt.title(f"Press_C X-bar chart (subgroups of {n})"); plt.ylabel("mean deg C"); plt.legend(); plt.show()
print("out-of-control subgroups:", len(bad))

## 3 · Process capability — Cp & Cpk
- **Cp** = (USL - LSL) / 6 sigma — is the spread narrow enough for the spec? (ignores centering)
- **Cpk** = min[(USL - mean), (mean - LSL)] / 3 sigma — capability accounting for how centred the process is.
A common target is **Cpk >= 1.33**.

In [ ]:
def capability(x):
    mu, sigma = x.mean(), x.std()
    cp  = (USL - LSL) / (6 * sigma)
    cpk = min((USL - mu) / (3 * sigma), (mu - LSL) / (3 * sigma))
    return round(mu, 1), round(sigma, 2), round(cp, 2), round(cpk, 2)

print(f"Spec: LSL={LSL}  target={TARGET}  USL={USL}\n")
print(f"{'machine':10}{'mean':>7}{'sigma':>7}{'Cp':>7}{'Cpk':>7}")
for m in ["Press_A", "Press_B", "Press_C"]:
    mu, sigma, cp, cpk = capability(df[df.machine==m]["oven_temp_c"])
    print(f"{m:10}{mu:>7}{sigma:>7}{cp:>7}{cpk:>7}")

## 4 · In control vs capable
Notice the machines can have similar **Cp** (same spread) but very different **Cpk** — Press_C runs hot, pushing its mean toward the USL, so it is far less capable even when its control chart looks stable.

**In control** (predictable, no special causes) is *not* the same as **capable** (consistently inside spec). You need both.

## Debrief
- Which machine is least capable, and is it a spread problem (Cp) or a centering problem (Cpk)?
- What is the concrete fix for a low-Cpk-but-ok-Cp process? (Re-centre it — here, bring Press_C temperature down.)
- SPC is the quality half of manufacturing analytics: control charts detect *when* something changed; capability tells you *whether the process can meet spec at all*.